# Final Assessment — Comprehensive Stock Price Research

**Fullstack AI Batch 11 — Practical / Programming Assessment**

This notebook follows the assessment framework:

**Data Validation → Market Structure Analysis → Feature Engineering → Statistical Research → Machine Learning → Deep Learning → Backtesting → Risk Analysis → Research Conclusions**

> Replace the CSV paths in the first code cell with the datasets supplied for the assessment before running the notebook.


## 1. Research Objective

The purpose of this project is not simply to predict tomorrow's stock price. The research investigates:

- What drives returns and volatility?
- Which technical indicators contain useful information?
- Whether trends continue or reverse.
- Whether market regimes can be detected.
- Whether machine-learning models generalize through time.
- Whether a probability-based strategy can produce useful risk-adjusted results.

The assessment emphasizes **NumPy, Pandas, Seaborn, Scikit-Learn, and TensorFlow/Keras**.


In [ ]:
# Cell 1 — Imports and configuration
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_squared_error, mean_absolute_error, classification_report
)

# TensorFlow / Keras is used for the deep-learning section.
try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Dense, LSTM, GRU, Conv1D, MaxPooling1D, Flatten, Dropout
    TENSORFLOW_AVAILABLE = True
except Exception:
    TENSORFLOW_AVAILABLE = False
    print("TensorFlow/Keras is not installed. Deep-learning cells will be skipped.")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42


In [ ]:
# Cell 2 — Dataset paths
# Update these paths to the CSV files provided for the assessment.

DATASETS = {
    "ADBE": "ADBE.csv",
    "MSFT": "MSFT.csv",
    "ORCL": "ORCL.csv",
    "CRM": "CRM.csv",
}

def load_stock_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Dataset not found: {path}\n"
            "Please place the CSV file in the notebook folder or update DATASETS."
        )

    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]

    # Normalize common column names.
    rename = {}
    for c in df.columns:
        key = c.lower().replace(" ", "").replace("_", "")
        mapping = {
            "date": "Date",
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "adjclose": "Adj Close",
            "volume": "Volume"
        }
        if key in mapping:
            rename[c] = mapping[key]

    df = df.rename(columns=rename)

    if "Date" not in df.columns or "Close" not in df.columns:
        raise ValueError("CSV must contain at least Date and Close columns.")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date", "Close"]).sort_values("Date")
    df = df.drop_duplicates(subset="Date").reset_index(drop=True)

    numeric_cols = [c for c in ["Open","High","Low","Close","Adj Close","Volume"] if c in df.columns]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["Close"]).copy()
    return df

# Load available datasets.
stocks = {}
for name, path in DATASETS.items():
    try:
        stocks[name] = load_stock_csv(path)
        print(f"{name}: {stocks[name].shape}")
    except Exception as e:
        print(f"{name}: {e}")


## 2. Data Validation and Descriptive Statistics

In [ ]:
# Cell 3 — Data validation
for name, df in stocks.items():
    print("\n" + "="*70)
    print(name)
    print("="*70)
    print("Shape:", df.shape)
    print("Date range:", df["Date"].min(), "to", df["Date"].max())
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nDuplicate dates:", df["Date"].duplicated().sum())
    print("\nDescriptive statistics:")
    display(df.describe(include="all").transpose())


In [ ]:
# Cell 4 — Core Pandas and NumPy features
def add_core_features(df):
    x = df.copy()
    x["Year"] = x["Date"].dt.year
    x["Month"] = x["Date"].dt.month
    x["Quarter"] = x["Date"].dt.quarter
    x["Day"] = x["Date"].dt.day

    x["Log_Return"] = np.log(x["Close"] / x["Close"].shift(1))
    x["Return"] = x["Close"].pct_change()

    x["MA20"] = x["Close"].rolling(20).mean()
    x["STD20"] = x["Close"].rolling(20).std()
    x["Mom5"] = x["Close"].pct_change(5)
    x["Mom20"] = x["Close"].pct_change(20)
    x["Vol20"] = x["Log_Return"].rolling(20).std() * np.sqrt(252)
    x["Volume_Ratio"] = x["Volume"] / x["Volume"].rolling(20).mean() if "Volume" in x.columns else np.nan

    return x

for name in list(stocks):
    stocks[name] = add_core_features(stocks[name])

display(stocks[next(iter(stocks))].tail())


In [ ]:
# Cell 5 — NumPy research metrics
def max_drawdown(prices):
    wealth = prices / prices.iloc[0]
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1
    return drawdown.min()

def annualized_volatility(log_returns):
    return log_returns.dropna().std() * np.sqrt(252)

def monte_carlo_final_returns(log_returns, n_simulations=1000, horizon=252):
    r = log_returns.dropna().to_numpy()
    if len(r) == 0:
        return np.array([])
    sampled = np.random.default_rng(RANDOM_STATE).choice(
        r, size=(n_simulations, horizon), replace=True
    )
    return np.exp(sampled.sum(axis=1)) - 1

for name, df in stocks.items():
    ann_vol = annualized_volatility(df["Log_Return"])
    mdd = max_drawdown(df["Close"])
    mc = monte_carlo_final_returns(df["Log_Return"])

    print(f"\n{name}")
    print(f"Annualized volatility: {ann_vol:.2%}")
    print(f"Maximum drawdown:     {mdd:.2%}")
    if len(mc):
        print(f"Monte Carlo median 1-year return: {np.median(mc):.2%}")
        print(f"Monte Carlo 5%-95% range: {np.percentile(mc,5):.2%} to {np.percentile(mc,95):.2%}")


## 3. Seaborn Exploratory Data Analysis

In [ ]:
# Cell 6 — Price and return distributions
for name, df in stocks.items():
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(df["Log_Return"].dropna(), kde=True, ax=ax)
    ax.set_title(f"{name} Log-Return Distribution")
    ax.set_xlabel("Log Return")
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.lineplot(data=df, x="Date", y="Close", ax=ax)
    ax.set_title(f"{name} Closing Price")
    plt.xticks(rotation=30)
    plt.show()


In [ ]:
# Cell 7 — Correlation heatmap
for name, df in stocks.items():
    numeric = df.select_dtypes(include=np.number)
    corr = numeric.corr()

    plt.figure(figsize=(12, 8))
    sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
    plt.title(f"{name} Feature Correlation Heatmap")
    plt.show()


In [ ]:
# Cell 8 — Pairplot using selected research variables
for name, df in stocks.items():
    cols = [c for c in ["Log_Return", "Mom5", "Mom20", "Vol20", "Volume_Ratio"] if c in df.columns]
    sample = df[cols].dropna().sample(min(500, len(df[cols].dropna())), random_state=RANDOM_STATE)
    if len(sample) >= 2:
        sns.pairplot(sample)
        plt.suptitle(f"{name} Pairplot", y=1.02)
        plt.show()


## 4. Technical Indicator Feature Engineering

In [ ]:
# Cell 9 — Technical indicators
def add_technical_indicators(df):
    x = df.copy()

    # Moving averages
    for w in [20, 50, 100, 200]:
        x[f"MA{w}"] = x["Close"].rolling(w).mean()

    # Volatility
    for w in [5, 10, 20, 60]:
        x[f"Vol{w}"] = x["Log_Return"].rolling(w).std() * np.sqrt(252)

    # High-low range and gap
    if {"High", "Low"}.issubset(x.columns):
        x["HL_Range"] = (x["High"] - x["Low"]) / x["Close"]
    if "Open" in x.columns:
        x["Gap"] = x["Open"] / x["Close"].shift(1) - 1

    # RSI
    delta = x["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    x["RSI"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = x["Close"].ewm(span=12, adjust=False).mean()
    ema26 = x["Close"].ewm(span=26, adjust=False).mean()
    x["MACD"] = ema12 - ema26
    x["MACD_Signal"] = x["MACD"].ewm(span=9, adjust=False).mean()
    x["MACD_Hist"] = x["MACD"] - x["MACD_Signal"]

    # ATR
    if {"High", "Low"}.issubset(x.columns):
        prev_close = x["Close"].shift(1)
        tr = pd.concat([
            x["High"] - x["Low"],
            (x["High"] - prev_close).abs(),
            (x["Low"] - prev_close).abs()
        ], axis=1).max(axis=1)
        x["ATR"] = tr.rolling(14).mean()

    # Bollinger Bands
    mid = x["Close"].rolling(20).mean()
    std = x["Close"].rolling(20).std()
    x["BB_Middle"] = mid
    x["BB_Upper"] = mid + 2 * std
    x["BB_Lower"] = mid - 2 * std
    x["BB_Width"] = (x["BB_Upper"] - x["BB_Lower"]) / mid

    # Stochastic oscillator
    if {"High", "Low"}.issubset(x.columns):
        low14 = x["Low"].rolling(14).min()
        high14 = x["High"].rolling(14).max()
        x["Stochastic_K"] = 100 * (x["Close"] - low14) / (high14 - low14)
        x["Stochastic_D"] = x["Stochastic_K"].rolling(3).mean()

    # OBV
    if "Volume" in x.columns:
        direction = np.sign(x["Close"].diff()).fillna(0)
        x["OBV"] = (direction * x["Volume"]).cumsum()

    # ADX
    if {"High", "Low"}.issubset(x.columns):
        up_move = x["High"].diff()
        down_move = -x["Low"].diff()
        plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
        minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)
        prev_close = x["Close"].shift(1)
        tr = pd.concat([
            x["High"] - x["Low"],
            (x["High"] - prev_close).abs(),
            (x["Low"] - prev_close).abs()
        ], axis=1).max(axis=1)
        atr14 = tr.rolling(14).mean()
        plus_di = 100 * pd.Series(plus_dm, index=x.index).rolling(14).mean() / atr14
        minus_di = 100 * pd.Series(minus_dm, index=x.index).rolling(14).mean() / atr14
        dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
        x["ADX"] = dx.rolling(14).mean()

    return x

for name in list(stocks):
    stocks[name] = add_technical_indicators(stocks[name])

display(stocks[next(iter(stocks))].tail())


## 5. Statistical Research and Market Regimes

In [ ]:
# Cell 10 — Regime labels for exploratory analysis
for name, df in stocks.items():
    q_low = df["Mom20"].rolling(252, min_periods=60).quantile(0.33)
    q_high = df["Mom20"].rolling(252, min_periods=60).quantile(0.67)

    df["Regime"] = np.select(
        [df["Mom20"] < q_low, df["Mom20"] > q_high],
        ["Bear", "Bull"],
        default="Sideways"
    )

    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df.dropna(subset=["Regime", "Log_Return"]), x="Regime", y="Log_Return")
    plt.title(f"{name} Returns by Market Regime")
    plt.show()


In [ ]:
# Cell 11 — KMeans clustering and PCA
for name, df in stocks.items():
    cols = ["Mom5", "Mom20", "Vol20", "RSI", "MACD_Hist"]
    available = [c for c in cols if c in df.columns]
    clean = df[available].dropna()

    if len(clean) < 30:
        continue

    scaler = StandardScaler()
    X = scaler.fit_transform(clean)

    kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X)

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    components = pca.fit_transform(X)

    plt.figure(figsize=(9, 6))
    plt.scatter(components[:,0], components[:,1], c=labels, alpha=0.6)
    plt.title(f"{name} KMeans Market Regimes — PCA Projection")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.show()

    print(f"{name} explained variance by 2 PCs: {pca.explained_variance_ratio_.sum():.2%}")


## 6. Machine Learning — Classification

**Target:** whether the next trading day's return is positive.

The assessment specifically calls for time-aware validation, so this notebook uses **TimeSeriesSplit** rather than random `train_test_split`.


In [ ]:
# Cell 12 — Prepare classification data
FEATURES = [
    "MA20", "MA50", "MA100", "MA200",
    "Vol5", "Vol10", "Vol20", "Vol60",
    "HL_Range", "Gap", "RSI", "MACD", "MACD_Signal",
    "ATR", "BB_Width", "Stochastic_K", "Stochastic_D",
    "OBV", "ADX", "Mom5", "Mom20", "Volume_Ratio"
]

def make_ml_data(df):
    x = df.copy()
    x["Target_Up"] = (x["Close"].shift(-1) > x["Close"]).astype(int)
    available = [c for c in FEATURES if c in x.columns]
    x = x.dropna(subset=available + ["Target_Up"]).copy()
    return x, available

ml_data = {}
for name, df in stocks.items():
    ml_data[name] = make_ml_data(df)
    print(name, ml_data[name][0].shape)


In [ ]:
# Cell 13 — Classification models with TimeSeriesSplit
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
    )
}

classification_results = []

for name, (df, features) in ml_data.items():
    X = df[features]
    y = df["Target_Up"]

    tscv = TimeSeriesSplit(n_splits=5)

    for model_name, model in models.items():
        scores = []
        for train_idx, test_idx in tscv.split(X):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            model.fit(X_train, y_train)
            pred = model.predict(X_test)

            scores.append({
                "Accuracy": accuracy_score(y_test, pred),
                "Precision": precision_score(y_test, pred, zero_division=0),
                "Recall": recall_score(y_test, pred, zero_division=0),
                "F1": f1_score(y_test, pred, zero_division=0)
            })

        avg = pd.DataFrame(scores).mean().to_dict()
        classification_results.append({
            "Stock": name,
            "Model": model_name,
            **avg
        })

classification_results = pd.DataFrame(classification_results)
display(classification_results.sort_values(["Stock", "F1"], ascending=[True, False]))


## 7. Regression Targets — Future Returns

The assessment recommends predicting **returns rather than raw prices**. The following targets estimate future 1-day, 5-day and 20-day returns.


In [ ]:
# Cell 14 — Future-return targets
for name, df in stocks.items():
    df["Future_1D_Return"] = df["Close"].shift(-1) / df["Close"] - 1
    df["Future_5D_Return"] = df["Close"].shift(-5) / df["Close"] - 1
    df["Future_20D_Return"] = df["Close"].shift(-20) / df["Close"] - 1

display(stocks[next(iter(stocks))][["Date","Close","Future_1D_Return","Future_5D_Return","Future_20D_Return"]].tail(10))


## 8. Deep Learning — TensorFlow / Keras

This section provides a reproducible sequence-model framework. It uses historical feature sequences to classify the next day's direction. If TensorFlow is unavailable, the cell reports that and continues with the classical ML results.


In [ ]:
# Cell 15 — LSTM/GRU sequence preparation
def make_sequences(df, features, lookback=20):
    clean = df.dropna(subset=features + ["Target_Up"]).reset_index(drop=True)
    X_raw = clean[features].to_numpy(dtype=np.float32)
    y_raw = clean["Target_Up"].to_numpy(dtype=np.float32)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    X_seq, y_seq = [], []
    for i in range(lookback, len(clean)):
        X_seq.append(X_scaled[i-lookback:i])
        y_seq.append(y_raw[i])

    return np.array(X_seq), np.array(y_seq)

if TENSORFLOW_AVAILABLE and len(ml_data):
    first_name = next(iter(ml_data))
    df0, feat0 = ml_data[first_name]
    X_seq, y_seq = make_sequences(df0, feat0, lookback=20)
    print("Sequence shape:", X_seq.shape, "Target shape:", y_seq.shape)
else:
    print("Deep-learning sequence preparation skipped.")


In [ ]:
# Cell 16 — LSTM and GRU models
def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, input_shape=input_shape),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_gru(input_shape):
    model = Sequential([
        GRU(64, input_shape=input_shape),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

if TENSORFLOW_AVAILABLE and len(ml_data) and len(X_seq) >= 100:
    split = int(len(X_seq) * 0.8)
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    lstm = build_lstm(X_train.shape[1:])
    history_lstm = lstm.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=10,
        batch_size=32,
        verbose=0
    )

    gru = build_gru(X_train.shape[1:])
    history_gru = gru.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=10,
        batch_size=32,
        verbose=0
    )

    lstm_prob = lstm.predict(X_test, verbose=0).ravel()
    gru_prob = gru.predict(X_test, verbose=0).ravel()

    print("LSTM accuracy:", accuracy_score(y_test, lstm_prob >= 0.5))
    print("GRU accuracy:", accuracy_score(y_test, gru_prob >= 0.5))
else:
    print("Not enough data or TensorFlow is unavailable.")


## 9. Feature Importance and Explainability

In [ ]:
# Cell 17 — Feature importance
importance_tables = []

for name, (df, features) in ml_data.items():
    X = df[features]
    y = df["Target_Up"]

    model = ExtraTreesClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
    )
    model.fit(X, y)

    imp = pd.DataFrame({
        "Feature": features,
        "Importance": model.feature_importances_
    }).sort_values("Importance", ascending=False)

    imp["Stock"] = name
    importance_tables.append(imp)

    plt.figure(figsize=(9, 7))
    sns.barplot(data=imp.head(15), x="Importance", y="Feature")
    plt.title(f"{name} — Top 15 Feature Importances")
    plt.show()

feature_importance = pd.concat(importance_tables, ignore_index=True) if importance_tables else pd.DataFrame()
display(feature_importance.head(30))


## 10. Probability-Based Backtesting

Strategy rule from the assessment:

**If predicted probability > 0.60 → BUY**

The backtest reports:

- CAGR
- Sharpe ratio
- Sortino ratio
- Maximum drawdown
- Calmar ratio


In [ ]:
# Cell 18 — Backtesting metrics
def sharpe_ratio(returns, periods=252):
    returns = pd.Series(returns).dropna()
    if returns.std() == 0:
        return np.nan
    return np.sqrt(periods) * returns.mean() / returns.std()

def sortino_ratio(returns, periods=252):
    returns = pd.Series(returns).dropna()
    downside = returns[returns < 0]
    downside_std = downside.std()
    if downside_std == 0 or pd.isna(downside_std):
        return np.nan
    return np.sqrt(periods) * returns.mean() / downside_std

def cagr(equity, periods=252):
    equity = pd.Series(equity).dropna()
    if len(equity) < 2 or equity.iloc[0] <= 0:
        return np.nan
    years = len(equity) / periods
    return (equity.iloc[-1] / equity.iloc[0]) ** (1 / years) - 1

def drawdown_series(equity):
    peak = equity.cummax()
    return equity / peak - 1

def calmar_ratio(equity):
    dd = drawdown_series(equity).min()
    cg = cagr(equity)
    return cg / abs(dd) if dd != 0 else np.nan

backtest_results = []

for name, (df, features) in ml_data.items():
    # Walk-forward out-of-sample predictions.
    X = df[features]
    y = df["Target_Up"]

    probs = pd.Series(index=df.index, dtype=float)
    tscv = TimeSeriesSplit(n_splits=5)

    for train_idx, test_idx in tscv.split(X):
        model = ExtraTreesClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        probs.iloc[test_idx] = model.predict_proba(X.iloc[test_idx])[:, 1]

    test = df.copy()
    test["Probability"] = probs
    test["Position"] = (test["Probability"] > 0.60).astype(int)

    # Next-day return is the tradable return after today's signal.
    test["Strategy_Return"] = test["Position"] * test["Future_1D_Return"]
    test = test.dropna(subset=["Strategy_Return"])

    equity = (1 + test["Strategy_Return"]).cumprod()
    metrics = {
        "Stock": name,
        "CAGR": cagr(equity),
        "Sharpe": sharpe_ratio(test["Strategy_Return"]),
        "Sortino": sortino_ratio(test["Strategy_Return"]),
        "Max Drawdown": drawdown_series(equity).min(),
        "Calmar": calmar_ratio(equity),
        "Trades/Active Days": int(test["Position"].sum())
    }
    backtest_results.append(metrics)

    plt.figure(figsize=(10, 5))
    sns.lineplot(x=test["Date"], y=equity)
    plt.title(f"{name} Probability Strategy Equity Curve")
    plt.ylabel("Growth of $1")
    plt.xticks(rotation=30)
    plt.show()

backtest_results = pd.DataFrame(backtest_results)
display(backtest_results)


## 11. Research Conclusions

After running the notebook on the supplied datasets, complete the conclusions using the actual results rather than invented values.

### Questions to answer

1. Which features had the highest importance?
2. Which stock showed the highest and lowest volatility?
3. Which market regimes were most persistent?
4. Did trend or momentum features help classify next-day direction?
5. Which classical ML model generalized best under TimeSeriesSplit?
6. Did LSTM/GRU improve on the classical baseline?
7. Did the probability > 0.60 strategy outperform a simple buy-and-hold benchmark?
8. What were the strategy's Sharpe, Sortino, CAGR, Calmar and maximum drawdown?
9. Where did the models fail?
10. What improvements should be tested in future research?

### Important research principles

- Predict returns rather than raw prices.
- Use time-aware validation.
- Compare against simple baselines.
- Avoid data leakage.
- Treat feature engineering and data quality as major parts of the research.
- Study failures, not only successful predictions.


## 12. Final Submission Checklist

- [ ] CSV datasets are correctly connected.
- [ ] Data validation has been completed.
- [ ] Descriptive statistics are included.
- [ ] NumPy metrics are calculated.
- [ ] Pandas features are created.
- [ ] Seaborn EDA plots are included.
- [ ] Technical indicators are included.
- [ ] Classification models are evaluated.
- [ ] TimeSeriesSplit is used.
- [ ] KMeans and PCA regime analysis is included.
- [ ] TensorFlow/Keras section is run where available.
- [ ] Feature importance is analyzed.
- [ ] Probability > 0.60 backtest is completed.
- [ ] Sharpe, Sortino, CAGR, Calmar and max drawdown are reported.
- [ ] Final conclusions are based on actual output.
- [ ] Notebook is saved with all cells executed before submission.
